# ImageGrains 2.0 — Sensitivity-Guided Mixed W4/W8 Quantization (Cellpose-SAM)

Follow-up to [`imagegrains_quantize_colab.ipynb`](imagegrains_quantize_colab.ipynb), testing the sensitivity-guided mixed W4/W8 scheme from [arXiv:2609.21038](https://arxiv.org/abs/2609.21038) (`MW4-SW8`) adapted for **`onnxruntime-web@1.31.1`**:

- **`fp16`**: Cached baseline (`ig2_cpsam_fp16.onnx`, 588.76 MiB).
- **`fp4_lut_mixed_fp8`**: 94/96 linear layers in `fp4_e2m1_lut` (4-bit), with the two sensitive ViT `Linear` layers (`blocks.0.mlp.lin2`, `blocks.23.attn.qkv`) kept in **`fp8_e4m3`** (`Conv`/`patch_embed`/`neck`/`out` remain in `fp16`).
- **`fp4_lut_mixed_fp16`**: Same 94/96 layers in `fp4_e2m1_lut`, with `blocks.0.mlp.lin2` and `blocks.23.attn.qkv` kept in **`fp16`**.
- **`int4_mnb_bs64_mixed`**: `MatMulNBits` (`block_size=64`) excluding `blocks.0.mlp.lin2` and `blocks.23.attn.qkv`.
- **`int8_wo_fp32scale`**: Fixed `INT8` weight-only (`DequantizeLinear(INT8 -> FLOAT32)` + `Cast(FLOAT32 -> FLOAT16)`) for a clean `W8A16` reference.

Reuses cached `fp32`/`fp16` ONNX models, `IG2_data.zip`, and `pt_ref_tiles` from `/content/drive/MyDrive/imagegrains_onnx_quant/` if already present.
Outputs `results_mixed_w4_w8.json` and `results_mixed_w4_w8_per_image.csv`.


In [ ]:
# CONFIG — Mixed W4/W8 follow-up for Cellpose-SAM
REPO_URL = "https://github.com/LukeAFullard/webGrainy.git"
REPO_BRANCH = "audit/quantization-fp8-fp4"

MODELS_TO_RUN = ["cpsam"]
VARIANTS_TO_RUN = [
    "fp16",
    "fp4_lut_mixed_fp8",
    "fp4_lut_mixed_fp16",
    "int4_mnb_bs64_mixed",
    "int8_wo_fp32scale",
]

EVAL_MAX_IMAGES = 40          # 40 = full IG2 official_split == 'test'
WASM_NUM_THREADS = 4          # Thread count for onnxruntime-web WASM in Node
SEED = 42

DRIVE_OUT_DIR = "/content/drive/MyDrive/imagegrains_onnx_quant"
DRIVE_DATA_DIR = "/content/drive/MyDrive/imagegrains_datasets"
LOCAL_WORK_DIR = "/content/quant_work"
SKIP_CACHED_ON_DRIVE = True   # Reuse cached models / checkpoints on Drive across Colab disconnects

SENSITIVE_LAYER_PATTERNS = (
    "blocks.0.mlp.lin2", "blocks.0/mlp/lin2",
    "blocks.23.attn.qkv", "blocks.23/attn/qkv",
)


In [ ]:
# Cell 1: Mount Drive, clone repo, install Node 22 + npm build, uv venvs (cp3 & cp4), and datasets/weights
import hashlib, os, psutil, shutil, subprocess, sys, urllib.request, zipfile
from tqdm.auto import tqdm

os.makedirs(LOCAL_WORK_DIR, exist_ok=True)
WEIGHTS_DIR = os.path.join(LOCAL_WORK_DIR, "weights")
DATA_DIR = os.path.join(LOCAL_WORK_DIR, "IG2_data")
MODELS_DIR = os.path.join(LOCAL_WORK_DIR, "models")
OUT_DIR = os.path.join(LOCAL_WORK_DIR, "outputs")
for d in [WEIGHTS_DIR, DATA_DIR, MODELS_DIR, OUT_DIR]:
    os.makedirs(d, exist_ok=True)

DRIVE_MOUNTED = False
try:
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")
    DRIVE_MOUNTED = os.path.exists("/content/drive/MyDrive")
except Exception as e:
    print("Drive mount skipped:", e)

if DRIVE_MOUNTED:
    os.makedirs(DRIVE_OUT_DIR, exist_ok=True)
    os.makedirs(os.path.join(DRIVE_OUT_DIR, "models"), exist_ok=True)
    os.makedirs(os.path.join(DRIVE_OUT_DIR, "cache"), exist_ok=True)

ram_gb = psutil.virtual_memory().total / (1024 ** 3)
disk_gb = shutil.disk_usage("/content").free / (1024 ** 3)
print(f"RAM: {ram_gb:.1f} GB | Free Disk: {disk_gb:.1f} GB | Drive mounted: {DRIVE_MOUNTED}")

# Ensure Node 22+ is available
def ensure_node22():
    try:
        v = subprocess.check_output(["node", "-v"], text=True).strip()
        major = int(v.lstrip("v").split(".")[0])
        if major >= 22:
            print("Node.js:", v)
            return
    except Exception:
        pass
    print("Installing Node.js 22...")
    subprocess.run("curl -fsSL https://deb.nodesource.com/setup_22.x | bash - && apt-get install -y -q nodejs", shell=True, check=True)
    print("Node.js:", subprocess.check_output(["node", "-v"], text=True).strip())

ensure_node22()

REPO_DIR = "/content/webGrainy"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
chk = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--verify", f"origin/{REPO_BRANCH}"], capture_output=True)
ref = f"origin/{REPO_BRANCH}" if chk.returncode == 0 else "origin/main"
subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", "quant_run", ref], check=True)
print("Repo checked out:", ref, subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"], text=True).strip())

subprocess.run(["npm", "ci", "--no-audit", "--no-fund"], cwd=REPO_DIR, check=True)
subprocess.run(["npm", "run", "build"], cwd=REPO_DIR, check=True)
subprocess.run(["node", "audit/followup/ts_postprocess_runner.mjs", "--selftest"], cwd=REPO_DIR, check=True)
subprocess.run(["node", "audit/followup/ts_cellpose_runner.mjs", "--selftest"], cwd=REPO_DIR, check=True)

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "uv", "onnx", "onnxscript", "onnxruntime-gpu", "ml_dtypes", "brotli",
    "scikit-image", "scipy", "pandas", "opencv-python-headless", "tqdm", "pillow",
], check=True)

def verify_venv(py_bin: str) -> bool:
    if not os.path.exists(py_bin):
        return False
    return subprocess.run([py_bin, "-c", "import numpy, torch, onnx, onnxruntime; from cellpose import models"], capture_output=True).returncode == 0

if not verify_venv("/content/venv_cp3/bin/python"):
    subprocess.run(["uv", "venv", "/content/venv_cp3", "--python", "3.11", "--clear", "-q"], check=True)
    subprocess.run(["uv", "pip", "install", "--python", "/content/venv_cp3/bin/python", "-q", "cellpose==3.1.1.1", "torch", "onnx", "onnxruntime", "numpy", "pillow"], check=True)

if not verify_venv("/content/venv_cp4/bin/python"):
    subprocess.run(["uv", "venv", "/content/venv_cp4", "--python", "3.11", "--clear", "-q"], check=True)
    subprocess.run(["uv", "pip", "install", "--python", "/content/venv_cp4/bin/python", "-q", "cellpose>=4.0.9,<4.2", "torch", "onnx", "onnxscript", "onnxruntime-gpu", "numpy", "pillow"], check=True)

def download_file(url: str, dst: str):
    if os.path.exists(dst) and os.path.getsize(dst) > 0:
        return
    if DRIVE_MOUNTED:
        for base in [DRIVE_DATA_DIR, os.path.join(DRIVE_DATA_DIR, "weights"), DRIVE_OUT_DIR, os.path.join(DRIVE_OUT_DIR, "models")]:
            cand = os.path.join(base, os.path.basename(dst))
            if os.path.exists(cand) and os.path.getsize(cand) > 0:
                print(f"Using Drive cache: {cand}")
                shutil.copy2(cand, dst)
                return
    print(f"Downloading {url} -> {dst}")
    req = urllib.request.Request(url, headers={"User-Agent": "webGrainy-quant/1.0"})
    with urllib.request.urlopen(req, timeout=240) as resp, open(dst, "wb") as f:
        total = int(resp.headers.get("Content-Length", 0))
        with tqdm(total=total, unit="B", unit_scale=True, desc=os.path.basename(dst)) as pbar:
            while True:
                chunk = resp.read(1024 * 1024)
                if not chunk:
                    break
                f.write(chunk)
                pbar.update(len(chunk))

download_file("https://zenodo.org/records/15728186/files/IG2_full_set.200525?download=1", os.path.join(WEIGHTS_DIR, "IG2_full_set.200525"))
download_file("https://zenodo.org/records/15728186/files/IG2_full_set_cp_SAM?download=1", os.path.join(WEIGHTS_DIR, "IG2_full_set_cp_SAM"))

import csv
MANIFEST_CSV = os.path.join(REPO_DIR, "audit/followup/heldout_manifest.csv")
with open(MANIFEST_CSV, newline="", encoding="utf-8") as f:
    manifest_rows = list(csv.DictReader(f))

if any(not os.path.exists(os.path.join(DATA_DIR, r["image_relpath"])) for r in manifest_rows):
    ig2_zip = os.path.join(LOCAL_WORK_DIR, "IG2_data.zip")
    download_file("https://zenodo.org/records/17866827/files/IG2_data.zip?download=1", ig2_zip)
    with zipfile.ZipFile(ig2_zip, "r") as zf:
        zf.extractall(DATA_DIR)

test_rows = [r for r in manifest_rows if r["official_split"] == "test"][:EVAL_MAX_IMAGES]
for r in test_rows:
    r["image_path"] = os.path.join(DATA_DIR, r["image_relpath"])
    r["mask_path"] = os.path.join(DATA_DIR, r["mask_relpath"])
print(f"Ready: {len(test_rows)} held-out test images.")


In [ ]:
# Cell 2: Export reference FP32 & baseline FP16 ONNX models + capture PyTorch FP32 reference tiles
import json, os, shutil, subprocess
import cv2
import numpy as np

# Build deterministic 23-tile suite:
# 20 real 256x256 tiles from held-out images + all-zero tile + 8-px zero-border tile + 100x80 zero-padded to 256x256
rng = np.random.default_rng(SEED)
TILES_DIR = os.path.join(LOCAL_WORK_DIR, "tiles")
os.makedirs(TILES_DIR, exist_ok=True)

def load_rgb(path: str) -> np.ndarray:
    bgr = cv2.imread(path, cv2.IMREAD_COLOR)
    if bgr is not None:
        return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    from PIL import Image
    return np.array(Image.open(path).convert("RGB"))

def norm_rgb_1_99(img_rgb: np.ndarray) -> np.ndarray:
    out = np.empty_like(img_rgb, dtype=np.float32)
    for c in range(3):
        ch = img_rgb[:, :, c].astype(np.float32)
        p1, p99 = np.percentile(ch, [1, 99])
        denom = p99 - p1
        out[:, :, c] = (ch - p1) / denom if denom > 1e-3 else 0.0
    return np.transpose(out, (2, 0, 1)) # [3, H, W]

def norm_gray_1_99(img_rgb: np.ndarray) -> np.ndarray:
    gray = img_rgb.astype(np.float32).mean(axis=2)
    p1, p99 = np.percentile(gray, [1, 99])
    denom = p99 - p1
    g = (gray - p1) / denom if denom > 1e-3 else np.zeros_like(gray)
    out = np.zeros((2, g.shape[0], g.shape[1]), dtype=np.float32)
    out[0] = g
    return out

def make_tiles(model_kind: str):
    C = 3 if model_kind == "cpsam" else 2
    tiles = []
    # 20 real 256x256 tiles from first 20 test images
    for idx, r in enumerate(test_rows[:20]):
        rgb = load_rgb(r["image_path"])
        arr = norm_rgb_1_99(rgb) if model_kind == "cpsam" else norm_gray_1_99(rgb)
        _, H, W = arr.shape
        if H < 256 or W < 256:
            pad_h, pad_w = max(0, 256 - H), max(0, 256 - W)
            arr = np.pad(arr, ((0, 0), (0, pad_h), (0, pad_w)), mode="constant")
            y0, x0 = 0, 0
        else:
            y0 = (H - 256) // 2
            x0 = (W - 256) // 2
        tiles.append((f"real_{idx:02d}_{r['uid'][:24]}", arr[:, y0:y0+256, x0:x0+256].astype(np.float32)))
    # 21: All-zero tile
    tiles.append(("all_zero", np.zeros((C, 256, 256), dtype=np.float32)))
    # 22: 8-px zero-border tile (get_pad_yx padding)
    t_b = tiles[0][1].copy()
    t_b[:, :8, :] = 0; t_b[:, -8:, :] = 0; t_b[:, :, :8] = 0; t_b[:, :, -8:] = 0
    tiles.append(("zero_border_8px", t_b))
    # 23: 100x80 image zero-padded to 256x256
    rgb_small = cv2.resize(load_rgb(test_rows[1]["image_path"]), (80, 100), interpolation=cv2.INTER_AREA)
    arr_small = norm_rgb_1_99(rgb_small) if model_kind == "cpsam" else norm_gray_1_99(rgb_small)
    t_pad = np.zeros((C, 256, 256), dtype=np.float32)
    t_pad[:, :100, :80] = arr_small
    tiles.append(("padded_100x80", t_pad))
    return tiles

export_unet_py = r'''
import argparse, json, os
import numpy as np
import onnx
import torch
from cellpose import models

class DynamicMakeStyle(torch.nn.Module):
    def forward(self, x0: torch.Tensor) -> torch.Tensor:
        style = x0.mean(dim=(-2, -1))
        style = style / torch.sum(style**2, dim=1, keepdim=True)**0.5
        return style

class CPnetWrapper(torch.nn.Module):
    def __init__(self, net: torch.nn.Module):
        super().__init__()
        self.net = net
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = self.net(x)
        return out[0] if isinstance(out, (tuple, list)) else out

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--weights", required=True)
    ap.add_argument("--fp32-out", required=True)
    ap.add_argument("--fp16-out", required=True)
    ap.add_argument("--tiles-npy", required=True)
    ap.add_argument("--pt-ref-out", required=True)
    args = ap.parse_args()

    cp_model = models.CellposeModel(gpu=False, pretrained_model=args.weights)
    net = cp_model.net.cpu().eval()
    net.mkldnn = False
    wrapper = CPnetWrapper(net).eval()

    tiles = np.load(args.tiles_npy)
    with torch.no_grad():
        refs = [wrapper(torch.from_numpy(t[None])).numpy()[0] for t in tiles]
    np.save(args.pt_ref_out, np.stack(refs, axis=0).astype(np.float32))

    net.make_style = DynamicMakeStyle()
    dummy = torch.rand(1, 2, 256, 256, dtype=torch.float32)
    if not os.path.exists(args.fp32_out):
        torch.onnx.export(
            wrapper, dummy, args.fp32_out, dynamo=False, export_params=True, opset_version=17,
            do_constant_folding=True, input_names=["image"], output_names=["flows_cellprob"],
            dynamic_axes={"image": {0: "batch", 2: "height", 3: "width"}, "flows_cellprob": {0: "batch", 2: "height", 3: "width"}},
        )
        onnx.checker.check_model(onnx.load(args.fp32_out))

    if not os.path.exists(args.fp16_out):
        from onnxruntime.transformers import float16
        m32 = onnx.load(args.fp32_out)
        m16 = float16.convert_float_to_float16(m32, keep_io_types=False)
        onnx.save(m16, args.fp16_out)
    print("U-Net export + PyTorch ref done.")

if __name__ == "__main__":
    main()
'''

export_cpsam_py = r'''
import argparse, os
import numpy as np
import onnx
import torch
from cellpose import models

class CPSAMWrapper(torch.nn.Module):
    def __init__(self, net: torch.nn.Module):
        super().__init__()
        self.net = net
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = self.net(x)
        return out[0] if isinstance(out, (tuple, list)) else out

def export_precision(weights, out_path, dtype, use_cuda):
    if os.path.exists(out_path):
        return
    device = torch.device("cuda" if use_cuda else "cpu")
    cp_model = models.CellposeModel(gpu=use_cuda, pretrained_model=weights)
    sam_net = cp_model.net
    sam_net.dtype = dtype
    sam_net = sam_net.to(dtype=dtype, device=device).eval()
    wrapper = CPSAMWrapper(sam_net).eval()
    trace_input = torch.rand(2, 3, 256, 256, dtype=dtype, device=device)
    batch_dim = torch.export.Dim("batch", min=1, max=32)
    try:
        ep = torch.export.export(wrapper, (trace_input,), dynamic_shapes={"x": {0: batch_dim}}, strict=True)
    except Exception:
        ep = torch.export.export(wrapper, (trace_input,), dynamic_shapes={"x": {0: batch_dim}}, strict=False)
    temp_path = out_path + ".tmp.onnx"
    prog = torch.onnx.export(ep, (trace_input,), dynamo=True, opset_version=18, input_names=["image"], output_names=["flows_cellprob"])
    prog.save(temp_path)
    proto = onnx.load(temp_path, load_external_data=True)
    onnx.save_model(proto, out_path, save_as_external_data=False)
    for f in (temp_path, temp_path + ".data"):
        if os.path.exists(f):
            os.remove(f)

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--weights", required=True)
    ap.add_argument("--fp32-out", required=True)
    ap.add_argument("--fp16-out", required=True)
    ap.add_argument("--tiles-npy", required=True)
    ap.add_argument("--pt-ref-out", required=True)
    args = ap.parse_args()

    use_cuda = torch.cuda.is_available()
    device = torch.device("cuda" if use_cuda else "cpu")
    cp_model = models.CellposeModel(gpu=use_cuda, pretrained_model=args.weights)
    sam_net = cp_model.net.float().to(device).eval()
    wrapper = CPSAMWrapper(sam_net).eval()

    tiles = np.load(args.tiles_npy)
    with torch.no_grad():
        refs = [wrapper(torch.from_numpy(t[None]).to(device)).float().cpu().numpy()[0] for t in tiles]
    np.save(args.pt_ref_out, np.stack(refs, axis=0).astype(np.float32))

    del cp_model, sam_net, wrapper
    if use_cuda:
        torch.cuda.empty_cache()

    export_precision(args.weights, args.fp32_out, torch.float32, use_cuda)
    export_precision(args.weights, args.fp16_out, torch.float16, use_cuda)
    print("CP-SAM FP32 + FP16 export + PyTorch ref done.")

if __name__ == "__main__":
    main()
'''

with open("/tmp/export_unet_quant.py", "w") as f:
    f.write(export_unet_py)
with open("/tmp/export_cpsam_quant.py", "w") as f:
    f.write(export_cpsam_py)

tile_names = {}
for m in MODELS_TO_RUN:
    tlist = make_tiles(m)
    tile_names[m] = [x[0] for x in tlist]
    tiles_npy = os.path.join(TILES_DIR, f"{m}_tiles.npy")
    np.save(tiles_npy, np.stack([x[1] for x in tlist], axis=0))
    pt_ref_npy = os.path.join(TILES_DIR, f"{m}_pt_fp32_ref.npy")
    fp32_p = os.path.join(MODELS_DIR, f"ig2_{m}_fp32.onnx")
    fp16_p = os.path.join(MODELS_DIR, f"ig2_{m}_fp16.onnx")
    if DRIVE_MOUNTED and SKIP_CACHED_ON_DRIVE:
        for p in [fp32_p, fp16_p]:
            dp = os.path.join(DRIVE_OUT_DIR, "models", os.path.basename(p))
            if os.path.exists(dp) and not os.path.exists(p):
                shutil.copy2(dp, p)
    py_bin = "/content/venv_cp4/bin/python" if m == "cpsam" else "/content/venv_cp3/bin/python"
    script = "/tmp/export_cpsam_quant.py" if m == "cpsam" else "/tmp/export_unet_quant.py"
    w_file = "IG2_full_set_cp_SAM" if m == "cpsam" else "IG2_full_set.200525"
    subprocess.run([
        py_bin, script,
        "--weights", os.path.join(WEIGHTS_DIR, w_file),
        "--fp32-out", fp32_p,
        "--fp16-out", fp16_p,
        "--tiles-npy", tiles_npy,
        "--pt-ref-out", pt_ref_npy,
    ], check=True)
    if DRIVE_MOUNTED:
        for p in [fp32_p, fp16_p]:
            dp = os.path.join(DRIVE_OUT_DIR, "models", os.path.basename(p))
            if not os.path.exists(dp):
                shutil.copy2(p, dp)


In [ ]:
# Cell 3: Quantize Mixed W4/W8 variants & fixed int8_wo_fp32scale
import brotli, gzip, hashlib, os, shutil
import ml_dtypes
import numpy as np
import onnx
from onnx import TensorProto, helper, numpy_helper
try:
    from onnxruntime.quantization.matmul_nbits_quantizer import MatMulNBitsQuantizer, DefaultWeightOnlyQuantConfig
except ImportError:
    from onnxruntime.quantization.matmul_4bits_quantizer import MatMul4BitsQuantizer as MatMulNBitsQuantizer, DefaultWeightOnlyQuantConfig

EXCLUDE_SUBSTRINGS = ("patch_embed", "conv", "norm", "head", "output")

def is_excluded(name: str) -> bool:
    low = name.lower()
    return any(s in low for s in EXCLUDE_SUBSTRINGS)

def is_sensitive_layer(node_name: str, w_name: str) -> bool:
    n_low = node_name.lower()
    w_low = w_name.lower()
    return any(p in n_low or p in w_low for p in SENSITIVE_LAYER_PATTERNS)

def model_weight_stats(model: onnx.ModelProto):
    inits = {i.name: i for i in model.graph.initializer}
    total_bytes = sum(len(numpy_helper.to_array(i).tobytes()) for i in model.graph.initializer)
    return total_bytes, inits

def find_target_weights(model: onnx.ModelProto, min_elems: int = 1024):
    inits = {i.name: i for i in model.graph.initializer}
    targets = []
    seen = set()
    for node in model.graph.node:
        if node.op_type not in ("MatMul", "Gemm"):
            continue
        if is_excluded(node.name):
            continue
        for inp in node.input[:2]:
            if inp not in inits or inp in seen or is_excluded(inp):
                continue
            arr = numpy_helper.to_array(inits[inp])
            if arr.ndim != 2 or arr.size < min_elems or arr.dtype not in (np.float16, np.float32):
                continue
            transB = 0
            if node.op_type == "Gemm":
                for attr in node.attribute:
                    if attr.name == "transB":
                        transB = attr.i
            out_axis = 0 if (node.op_type == "Gemm" and transB == 1) else 1
            k_axis = 1 if (node.op_type == "Gemm" and transB == 1) else 0
            targets.append((node.name, inp, out_axis, k_axis))
            seen.add(inp)
    return targets

def remove_init(model: onnx.ModelProto, name: str):
    for i, init in enumerate(model.graph.initializer):
        if init.name == name:
            model.graph.initializer.pop(i)
            return

def quantize_fp4_lut_mixed(src_path: str, dst_path: str, base_dtype: int, sensitive_mode: str = "fp8", block_size: int = 32):
    assert sensitive_mode in ("fp8", "fp16")
    model = onnx.load(src_path)
    min_opset = 19 if sensitive_mode == "fp8" else 18
    model.opset_import[0].version = max(model.opset_import[0].version, min_opset)
    if sensitive_mode == "fp8":
        model.ir_version = max(model.ir_version, 9)
    total_bytes, inits = model_weight_stats(model)
    targets = find_target_weights(model)
    np_dt = np.float16 if base_dtype == TensorProto.FLOAT16 else np.float32

    lut16 = np.arange(16, dtype=np.uint8).view(ml_dtypes.float4_e2m1fn).astype(np.float32)
    lut256 = np.zeros((256, 2), dtype=np_dt)
    for b in range(256):
        lut256[b, 0] = lut16[b & 0x0F]
        lut256[b, 1] = lut16[(b >> 4) & 0xF]
    model.graph.initializer.append(numpy_helper.from_array(lut256, "shared_fp4_lut256"))

    q_bytes = 0
    q_layers = []
    sensitive_matched = []
    new_nodes = []
    for node_name, w_name, out_axis, k_axis in targets:
        W = numpy_helper.to_array(inits[w_name]).astype(np.float32)
        R, C = W.shape
        if is_sensitive_layer(node_name, w_name):
            sensitive_matched.append((node_name, w_name, (R, C)))
            if sensitive_mode == "fp16":
                continue
            # Keep sensitive layer in fp8_e4m3
            q_bytes += W.size * (2 if base_dtype == TensorProto.FLOAT16 else 4)
            q_layers.append(f"{w_name}[fp8]")
            red_axis = 1 - out_axis
            amax = np.max(np.abs(W), axis=red_axis)
            scales = np.where(amax == 0, 1.0, amax / 448.0).astype(np_dt)
            s_bcast = scales[:, None] if out_axis == 0 else scales[None, :]
            W_q = (W / s_bcast.astype(np.float32)).astype(ml_dtypes.float8_e4m3fn)
            remove_init(model, w_name)
            model.graph.initializer.extend([
                helper.make_tensor(w_name + "_fp8", TensorProto.FLOAT8E4M3FN, W_q.shape, W_q.tobytes(), raw=True),
                numpy_helper.from_array(scales, w_name + "_scale"),
            ])
            new_nodes.append(helper.make_node("DequantizeLinear", [w_name + "_fp8", w_name + "_scale"], [w_name], axis=out_axis))
            continue

        if W.size % 2 != 0:
            continue
        k_len = W.shape[k_axis]
        bs = block_size if (k_len % block_size == 0) else k_len
        n_blocks = k_len // bs
        if k_axis == 0:
            Wb = W.reshape(n_blocks, bs, C)
            amax = np.max(np.abs(Wb), axis=1, keepdims=True)
            scales = np.where(amax == 0, 1.0, amax / 6.0).astype(np_dt)
            W_f4 = (Wb / scales.astype(np.float32)).astype(ml_dtypes.float4_e2m1fn).reshape(R, C)
            b_shape = np.array([n_blocks, bs, C], dtype=np.int64)
        else:
            Wb = W.reshape(R, n_blocks, bs)
            amax = np.max(np.abs(Wb), axis=2, keepdims=True)
            scales = np.where(amax == 0, 1.0, amax / 6.0).astype(np_dt)
            W_f4 = (Wb / scales.astype(np.float32)).astype(ml_dtypes.float4_e2m1fn).reshape(R, C)
            b_shape = np.array([R, n_blocks, bs], dtype=np.int64)

        nibbles = W_f4.view(np.uint8).reshape(-1) & 0x0F
        packed = (nibbles[0::2] | (nibbles[1::2] << 4)).astype(np.uint8)
        q_bytes += W.size * (2 if base_dtype == TensorProto.FLOAT16 else 4)
        q_layers.append(w_name)
        remove_init(model, w_name)
        model.graph.initializer.extend([
            numpy_helper.from_array(packed, w_name + "_fp4_u8"),
            numpy_helper.from_array(b_shape, w_name + "_bshape"),
            numpy_helper.from_array(scales, w_name + "_bscale"),
            numpy_helper.from_array(np.array([R, C], dtype=np.int64), w_name + "_shape"),
        ])
        new_nodes.extend([
            helper.make_node("Cast", [w_name + "_fp4_u8"], [w_name + "_idx"], to=TensorProto.INT32),
            helper.make_node("Gather", ["shared_fp4_lut256", w_name + "_idx"], [w_name + "_pairs"], axis=0),
            helper.make_node("Reshape", [w_name + "_pairs", w_name + "_bshape"], [w_name + "_blk"]),
            helper.make_node("Mul", [w_name + "_blk", w_name + "_bscale"], [w_name + "_scaled"]),
            helper.make_node("Reshape", [w_name + "_scaled", w_name + "_shape"], [w_name]),
        ])

    print(f"  [mixed {sensitive_mode}] matched {len(sensitive_matched)} sensitive layers: {sensitive_matched}")
    for n in reversed(new_nodes):
        model.graph.node.insert(0, n)
    onnx.save_model(model, dst_path, save_as_external_data=False)
    return q_layers, q_bytes, total_bytes

def quantize_int8_wo_fp32scale(src_path: str, dst_path: str, base_dtype: int):
    # ORT-Web 1.31.1 requires float32 scale for INT8 DequantizeLinear; Cast back to fp16 if needed.
    model = onnx.load(src_path)
    model.opset_import[0].version = max(model.opset_import[0].version, 18)
    total_bytes, inits = model_weight_stats(model)
    targets = find_target_weights(model)
    q_bytes = 0
    q_layers = []
    new_nodes = []
    for _, w_name, out_axis, _ in targets:
        W = numpy_helper.to_array(inits[w_name]).astype(np.float32)
        q_bytes += W.size * (2 if base_dtype == TensorProto.FLOAT16 else 4)
        q_layers.append(w_name)
        red_axis = 1 - out_axis
        amax = np.max(np.abs(W), axis=red_axis)
        scales = np.where(amax == 0, 1.0, amax / 127.0).astype(np.float32)
        s_bcast = scales[:, None] if out_axis == 0 else scales[None, :]
        W_q = np.clip(np.round(W / s_bcast), -127, 127).astype(np.int8)
        remove_init(model, w_name)
        model.graph.initializer.extend([
            numpy_helper.from_array(W_q, w_name + "_i8"),
            numpy_helper.from_array(scales, w_name + "_scale"),
        ])
        if base_dtype == TensorProto.FLOAT16:
            dq_out = w_name + "_dq_f32"
            new_nodes.append(helper.make_node("DequantizeLinear", [w_name + "_i8", w_name + "_scale"], [dq_out], axis=out_axis))
            new_nodes.append(helper.make_node("Cast", [dq_out], [w_name], to=TensorProto.FLOAT16))
        else:
            new_nodes.append(helper.make_node("DequantizeLinear", [w_name + "_i8", w_name + "_scale"], [w_name], axis=out_axis))
    for n in reversed(new_nodes):
        model.graph.node.insert(0, n)
    onnx.save_model(model, dst_path, save_as_external_data=False)
    return q_layers, q_bytes, total_bytes

def quantize_int4_mnb_mixed(src_path: str, dst_path: str, block_size: int = 64):
    src_m = onnx.load(src_path)
    total_bytes, inits = model_weight_stats(src_m)
    nodes_to_exclude = []
    for n in src_m.graph.node:
        if n.op_type == "MatMul":
            if is_excluded(n.name) or any(is_sensitive_layer(n.name, inp) for inp in n.input):
                nodes_to_exclude.append(n.name)
    print(f"  [int4_mnb_bs{block_size}_mixed] excluding {len(nodes_to_exclude)} MatMul nodes")
    cfg = DefaultWeightOnlyQuantConfig(block_size=block_size, is_symmetric=True, accuracy_level=4)
    q = MatMulNBitsQuantizer(src_m, nodes_to_exclude=nodes_to_exclude, algo_config=cfg)
    q.process()
    q.model.save_model_to_file(dst_path, use_external_data_format=False)
    q_m = onnx.load(dst_path)
    mnb_nodes = [n for n in q_m.graph.node if n.op_type == "MatMulNBits"]
    q_bytes = 0
    for n in mnb_nodes:
        if len(n.input) >= 2 and n.input[1] in q_m.graph.initializer.__str__():
            pass
    q_inits = {i.name: i for i in q_m.graph.initializer}
    for n in mnb_nodes:
        if len(n.input) >= 2 and n.input[1] in q_inits:
            arr = numpy_helper.to_array(q_inits[n.input[1]])
            q_bytes += arr.size * 2 * 2  # packed 4-bit -> fp16 bytes
    return [n.name for n in mnb_nodes], q_bytes, total_bytes

def file_metrics(path: str):
    raw = open(path, "rb").read()
    return {
        "bytes": len(raw),
        "mb": round(len(raw) / (1024 * 1024), 2),
        "gzip_bytes": len(gzip.compress(raw, compresslevel=6)),
        "gzip_mb": round(len(gzip.compress(raw, compresslevel=6)) / (1024 * 1024), 2),
        "brotli_bytes": len(brotli.compress(raw, quality=6)),
        "brotli_mb": round(len(brotli.compress(raw, quality=6)) / (1024 * 1024), 2),
        "sha256": hashlib.sha256(raw).hexdigest(),
    }

DRIVE_MODELS_DIR = os.path.join(DRIVE_OUT_DIR, "models")
if DRIVE_MOUNTED:
    os.makedirs(DRIVE_MODELS_DIR, exist_ok=True)

model_registry = {}
for m in MODELS_TO_RUN:
    model_registry[m] = {}
    base_fp16 = os.path.join(MODELS_DIR, f"ig2_{m}_fp16.onnx")
    for v in VARIANTS_TO_RUN:
        dst = os.path.join(MODELS_DIR, f"ig2_{m}_{v}.onnx")
        drive_dst = os.path.join(DRIVE_MODELS_DIR, f"ig2_{m}_{v}.onnx") if DRIVE_MOUNTED else None
        q_layers, q_bytes, total_bytes = [], 0, 0
        if v == "fp16":
            shutil.copy2(base_fp16, dst) if not os.path.exists(dst) else None
        else:
            if DRIVE_MOUNTED and SKIP_CACHED_ON_DRIVE and os.path.exists(drive_dst) and not os.path.exists(dst):
                shutil.copy2(drive_dst, dst)
            if not os.path.exists(dst):
                print(f"Building {m}/{v}...")
                if v == "fp4_lut_mixed_fp8":
                    q_layers, q_bytes, total_bytes = quantize_fp4_lut_mixed(base_fp16, dst, TensorProto.FLOAT16, sensitive_mode="fp8", block_size=32)
                elif v == "fp4_lut_mixed_fp16":
                    q_layers, q_bytes, total_bytes = quantize_fp4_lut_mixed(base_fp16, dst, TensorProto.FLOAT16, sensitive_mode="fp16", block_size=32)
                elif v == "int4_mnb_bs64_mixed":
                    q_layers, q_bytes, total_bytes = quantize_int4_mnb_mixed(base_fp16, dst, block_size=64)
                elif v == "int8_wo_fp32scale":
                    q_layers, q_bytes, total_bytes = quantize_int8_wo_fp32scale(base_fp16, dst, TensorProto.FLOAT16)
                if DRIVE_MOUNTED:
                    shutil.copy2(dst, drive_dst)
        sz = file_metrics(dst)
        model_registry[m][v] = {
            "path": dst,
            "size": sz,
            "quantized_layer_count": len(q_layers),
            "quantized_weight_fraction": round(q_bytes / total_bytes, 4) if total_bytes else 0.0,
        }
        print(f"  {m}/{v}: {sz['mb']} MiB (gz {sz['gzip_mb']}, br {sz['brotli_mb']}), q_layers={len(q_layers)}, sha256={sz['sha256'][:12]}")


In [ ]:
# Cell 4: Test 1 (ORT-Web 1.31.1 WASM load), Test 2 (23-tile numerics vs PyTorch FP32), Test 4 (Speed)
import json, os, subprocess, tempfile, time
import numpy as np
import onnxruntime as ort

wasm_bench_js = r'''
import fs from "node:fs";
import path from "node:path";
import { pathToFileURL } from "node:url";

const ortDist = path.join(process.cwd(), "node_modules/onnxruntime-web/dist");
const ort = await import(pathToFileURL(path.join(ortDist, "ort.min.mjs")).href);
const meta = JSON.parse(fs.readFileSync(process.argv[2], "utf8"));

ort.env.wasm.numThreads = Number(meta.numThreads || 4);
ort.env.wasm.simd = true;
ort.env.wasm.wasmPaths = ortDist + "/";

const rssMb = () => Math.round(process.memoryUsage().rss / (1024 * 1024) * 10) / 10;
let peakRss = rssMb();

const t0 = performance.now();
let session;
try {
  const bytes = new Uint8Array(fs.readFileSync(meta.modelPath));
  session = await ort.InferenceSession.create(bytes, { executionProviders: ["wasm"] });
  peakRss = Math.max(peakRss, rssMb());
} catch (err) {
  fs.writeFileSync(meta.resultJson, JSON.stringify({
    load_ok: false,
    load_error: String(err?.message || err),
    load_time_s: (performance.now() - t0) / 1000,
    peak_rss_mb: Math.max(peakRss, rssMb()),
  }));
  process.exit(0);
}
const loadTimeS = (performance.now() - t0) / 1000;

function f32ToF16Bits(f32) {
  const buf = new ArrayBuffer(4);
  const fv = new Float32Array(buf);
  const uv = new Uint32Array(buf);
  const out = new Uint16Array(f32.length);
  for (let i = 0; i < f32.length; i++) {
    fv[0] = f32[i];
    const b = uv[0];
    const s = (b >>> 16) & 0x8000;
    const e = (b >>> 23) & 0xff;
    const m = b & 0x7fffff;
    if (e === 0xff) { out[i] = s | 0x7c00 | (m ? 0x0200 : 0); continue; }
    if (e === 0) { out[i] = s; continue; }
    const ee = e - 127;
    if (ee > 15) { out[i] = s | 0x7c00; continue; }
    if (ee >= -14) {
      let he = ee + 15, hm = m >>> 13, rem = m & 0x1fff;
      if (rem > 0x1000 || (rem === 0x1000 && (hm & 1))) { hm++; if (hm === 0x400) { hm = 0; he++; } }
      out[i] = he >= 31 ? (s | 0x7c00) : (s | (he << 10) | hm);
    } else { out[i] = s; }
  }
  return out;
}

const raw = fs.readFileSync(meta.tilesPath);
const f32All = new Float32Array(raw.buffer, raw.byteOffset, raw.byteLength / 4);
const [N, C, H, W] = meta.tilesShape;
const tileElems = C * H * W;
const outTileElems = 3 * H * W;
const allOut = new Float32Array(N * outTileElems);
const timesMs = [];

try {
  for (let i = 0; i < N; i++) {
    const slice = f32All.subarray(i * tileElems, (i + 1) * tileElems);
    const tensor = meta.inputDtype === "float16"
      ? new ort.Tensor("float16", f32ToF16Bits(slice), [1, C, H, W])
      : new ort.Tensor("float32", slice, [1, C, H, W]);
    const r0 = performance.now();
    const res = await session.run({ image: tensor });
    timesMs.push(performance.now() - r0);
    peakRss = Math.max(peakRss, rssMb());
    const d = res.flows_cellprob.data;
    if (d instanceof Float32Array) {
      allOut.set(d, i * outTileElems);
    } else if (d instanceof Uint16Array) {
      for (let j = 0; j < outTileElems; j++) {
        const u = d[j], s = (u & 0x8000) << 16, e = (u >>> 10) & 0x1f, m = u & 0x3ff;
        const buf = new ArrayBuffer(4);
        const uv = new Uint32Array(buf), fv = new Float32Array(buf);
        if (e === 0) uv[0] = s;
        else if (e === 31) uv[0] = s | 0x7f800000 | (m << 13);
        else uv[0] = s | ((e + 112) << 23) | (m << 13);
        allOut[i * outTileElems + j] = fv[0];
      }
    } else {
      for (let j = 0; j < outTileElems; j++) allOut[i * outTileElems + j] = Number(d[j]);
    }
  }
  fs.writeFileSync(meta.outPath, Buffer.from(allOut.buffer, allOut.byteOffset, allOut.byteLength));
  const avgMs = timesMs.slice(1).reduce((a, b) => a + b, 0) / Math.max(1, timesMs.length - 1);
  fs.writeFileSync(meta.resultJson, JSON.stringify({
    load_ok: true,
    run_ok: true,
    load_time_s: loadTimeS,
    peak_rss_mb: peakRss,
    avg_tile_ms: avgMs,
  }));
} catch (err) {
  fs.writeFileSync(meta.resultJson, JSON.stringify({
    load_ok: true,
    run_ok: false,
    run_error: String(err?.message || err),
    load_time_s: loadTimeS,
    peak_rss_mb: Math.max(peakRss, rssMb()),
  }));
}
'''

with open("/tmp/wasm_quant_bench.mjs", "w") as f:
    f.write(wasm_bench_js)

def bench_py_ort(model_path: str, tiles: np.ndarray, input_dtype: str, provider: str):
    if provider == "CUDAExecutionProvider" and "CUDAExecutionProvider" not in ort.get_available_providers():
        return None, None
    try:
        sess = ort.InferenceSession(model_path, providers=[provider])
        arr = tiles.astype(np.float16 if input_dtype == "float16" else np.float32)
        times = []
        outs = []
        for i in range(len(arr)):
            t0 = time.perf_counter()
            o = sess.run(["flows_cellprob"], {"image": arr[i:i+1]})[0][0].astype(np.float32)
            times.append((time.perf_counter() - t0) * 1000.0)
            outs.append(o)
        return float(np.mean(times[1:])), np.stack(outs, axis=0)
    except Exception as e:
        return None, str(e)

tile_results = {}
for (m, v), info in variant_meta.items():
    tiles = np.load(os.path.join(TILES_DIR, f"{m}_tiles.npy"))
    pt_ref = np.load(os.path.join(TILES_DIR, f"{m}_pt_fp32_ref.npy"))
    cache_json = os.path.join(OUT_DIR, f"tile_bench_{m}_{v}.json")
    drive_cache = os.path.join(DRIVE_OUT_DIR, "cache", f"tile_bench_{m}_{v}.json") if DRIVE_MOUNTED else None
    if DRIVE_MOUNTED and SKIP_CACHED_ON_DRIVE and os.path.exists(drive_cache):
        shutil.copy2(drive_cache, cache_json)

    with tempfile.TemporaryDirectory() as td:
        tiles_p = os.path.join(td, "tiles.f32")
        out_p = os.path.join(td, "out.f32")
        res_p = os.path.join(td, "res.json")
        meta_p = os.path.join(td, "meta.json")
        np.ascontiguousarray(tiles, dtype="<f4").tofile(tiles_p)
        with open(meta_p, "w") as f:
            json.dump({
                "modelPath": info["path"],
                "tilesPath": tiles_p,
                "tilesShape": list(tiles.shape),
                "inputDtype": info["input_dtype"],
                "outPath": out_p,
                "resultJson": res_p,
                "numThreads": WASM_NUM_THREADS,
            }, f)
        subprocess.run(["node", "/tmp/wasm_quant_bench.mjs", meta_p], cwd=REPO_DIR, check=True)
        wasm_res = json.load(open(res_p))
        wasm_out = np.fromfile(out_p, dtype="<f4").reshape(pt_ref.shape) if wasm_res.get("run_ok") else None

    py_cpu_ms, py_cpu_out = bench_py_ort(info["path"], tiles, info["input_dtype"], "CPUExecutionProvider")
    py_cuda_ms, py_cuda_out = bench_py_ort(info["path"], tiles, info["input_dtype"], "CUDAExecutionProvider")

    numerics = None
    bridge_matches_wasm = False
    eval_out = wasm_out if wasm_out is not None else (py_cpu_out if isinstance(py_cpu_out, np.ndarray) else None)
    if eval_out is not None:
        non_finite = int(np.sum(~np.isfinite(eval_out)))
        diff = np.abs(eval_out - pt_ref)
        # All-zero tile index is 20
        az_cp = eval_out[20, 2]
        az_ref = pt_ref[20, 2]
        az_sign_match = float(100.0 * np.mean((az_cp < 0) == (az_ref < 0)))
        all_sign_match = float(100.0 * np.mean((eval_out[:, 2] < 0) == (pt_ref[:, 2] < 0)))
        numerics = {
            "source": "wasm" if wasm_out is not None else "python_cpu",
            "non_finite_count": non_finite,
            "flowY_max_abs": float(np.nanmax(diff[:, 0])), "flowY_mean_abs": float(np.nanmean(diff[:, 0])),
            "flowX_max_abs": float(np.nanmax(diff[:, 1])), "flowX_mean_abs": float(np.nanmean(diff[:, 1])),
            "cellprob_max_abs": float(np.nanmax(diff[:, 2])), "cellprob_mean_abs": float(np.nanmean(diff[:, 2])),
            "cellprob_sign_agree_pct": all_sign_match,
            "all_zero_cellprob_mean": float(np.nanmean(az_cp)),
            "all_zero_cellprob_max": float(np.nanmax(az_cp)),
            "all_zero_sign_agree_pct": az_sign_match,
            "all_zero_negative_ok": bool(np.all(az_cp < 0) and az_sign_match >= 99.0),
        }
        if wasm_out is not None and isinstance(py_cpu_out, np.ndarray):
            bridge_max_diff = float(np.max(np.abs(wasm_out - py_cpu_out)))
            numerics["py_cpu_vs_wasm_max_abs"] = bridge_max_diff
            bridge_matches_wasm = bridge_max_diff < 0.05

    rec = {
        "wasm": wasm_res,
        "speed_ms": {
            "wasm_threads": WASM_NUM_THREADS,
            "wasm_ms": wasm_res.get("avg_tile_ms"),
            "py_cpu_ms": py_cpu_ms,
            "py_cuda_ms": py_cuda_ms,
        },
        "numerics_vs_pt_fp32": numerics,
        "bridge_matches_wasm": bridge_matches_wasm,
    }
    tile_results[(m, v)] = rec
    with open(cache_json, "w") as f:
        json.dump(rec, f, indent=2)
    if DRIVE_MOUNTED:
        shutil.copy2(cache_json, drive_cache)

    status = "RUN_OK" if wasm_res.get("run_ok") else f"FAIL({wasm_res.get('load_error') or wasm_res.get('run_error')})"
    az_str = f"az_cp={numerics['all_zero_cellprob_mean']:.2f}" if numerics else "n/a"
    print(f"[{m:5s} {v:14s}] ORT-Web: {status[:48]} | {az_str} | wasm={rec['speed_ms']['wasm_ms']}ms cpu={py_cpu_ms}ms cuda={py_cuda_ms}ms")


In [ ]:
# Cell 5: Test 3 — Full 40-image held-out evaluation via ts_cellpose_runner.mjs + ts_postprocess_runner.mjs
import http.server, math, os, shutil, socketserver, subprocess, tempfile, threading
import numpy as np
import onnxruntime as ort
import pandas as pd
from skimage.measure import regionprops
from tqdm.auto import tqdm

TS_CELLPOSE_RUNNER = os.path.join(REPO_DIR, "audit/followup/ts_cellpose_runner.mjs")
TS_POSTPROCESS_RUNNER = os.path.join(REPO_DIR, "audit/followup/ts_postprocess_runner.mjs")

def load_gt_mask(path: str, hw: tuple[int, int]) -> np.ndarray:
    arr = np.load(path) if path.endswith(".npy") else cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if arr.ndim == 3:
        arr = arr[:, :, 0] if np.array_equal(arr[:, :, 0], arr[:, :, 1]) else ((arr[:, :, 0].astype(np.int64) << 16) | (arr[:, :, 1].astype(np.int64) << 8) | arr[:, :, 2])
    arr_i = arr.astype(np.int64); arr_i[arr_i < 0] = 0
    uniq, inv = np.unique(arr_i, return_inverse=True)
    out = inv.reshape(hw).astype(np.int32)
    return out + 1 if (len(uniq) > 0 and uniq[0] != 0) else out

def compute_mask_grain_metrics(masks: np.ndarray, margin: int = 2) -> list[dict]:
    H, W = masks.shape
    grains = []
    for prop in regionprops(masks):
        area = float(prop.area)
        if area <= 0:
            continue
        minr, minc, maxr, maxc = prop.bbox
        if minr < margin or minc < margin or maxr > H - margin or maxc > W - margin:
            continue
        b_axis = float(prop.axis_minor_length) if hasattr(prop, "axis_minor_length") else float(prop.minor_axis_length)
        d_eq = math.sqrt(4.0 * area / math.pi)
        if b_axis > 0 and d_eq > 0:
            grains.append({"areaPx": area, "bAxisPx": b_axis, "dEqPx": d_eq})
    return grains

def area_weighted_percentiles(grains: list[dict], key: str, pcts=(16.0, 50.0, 84.0)) -> dict[str, float]:
    valid = [(float(g[key]), float(g["areaPx"])) for g in grains if not g.get("touchesBorder", g.get("isBorder", False)) and g.get(key, 0) > 0 and g.get("areaPx", 0) > 0]
    if not valid:
        return {f"D{int(p)}": float("nan") for p in pcts}
    valid.sort(key=lambda x: x[0])
    vals = np.array([v[0] for v in valid], dtype=np.float64)
    wts = np.array([v[1] for v in valid], dtype=np.float64)
    cdf = (np.cumsum(wts) - 0.5 * wts) / np.sum(wts) * 100.0
    return {f"D{int(p)}": float(np.interp(p, cdf, vals)) for p in pcts}

def compute_pq(pred: np.ndarray, gt: np.ndarray, iou_thresh: float = 0.5) -> float:
    pred_ids = np.unique(pred); pred_ids = pred_ids[pred_ids > 0]
    gt_ids = np.unique(gt); gt_ids = gt_ids[gt_ids > 0]
    if len(pred_ids) == 0 and len(gt_ids) == 0: return 1.0
    if len(pred_ids) == 0 or len(gt_ids) == 0: return 0.0
    p_areas = {int(i): int((pred == i).sum()) for i in pred_ids}
    g_areas = {int(j): int((gt == j).sum()) for j in gt_ids}
    mask = (pred > 0) & (gt > 0)
    if not np.any(mask): return 0.0
    pairs = pred[mask].astype(np.int64) * 1_000_000 + gt[mask].astype(np.int64)
    uniq_pairs, counts = np.unique(pairs, return_counts=True)
    candidates = []
    for code, inter in zip(uniq_pairs, counts):
        pi, gi = int(code // 1_000_000), int(code % 1_000_000)
        iou = float(inter) / float(p_areas[pi] + g_areas[gi] - int(inter))
        if iou > iou_thresh:
            candidates.append((iou, pi, gi))
    candidates.sort(reverse=True)
    tp, iou_sum, mp, mg = 0, 0.0, set(), set()
    for iou, pi, gi in candidates:
        if pi not in mp and gi not in mg:
            mp.add(pi); mg.add(gi); tp += 1; iou_sum += iou
    denom = tp + 0.5 * (len(pred_ids) - tp) + 0.5 * (len(gt_ids) - tp)
    return float(iou_sum / denom) if denom > 0 else 0.0

class _ThreadingHTTPServer(socketserver.ThreadingMixIn, http.server.HTTPServer):
    daemon_threads = True

def start_ort_bridge(sess: ort.InferenceSession):
    lock = threading.Lock()
    class Handler(http.server.BaseHTTPRequestHandler):
        protocol_version = "HTTP/1.1"
        def log_message(self, *args): pass
        def do_POST(self):
            length = int(self.headers["Content-Length"])
            body = self.rfile.read(length)
            dt = np.float16 if self.headers["x-dtype"] == "float16" else np.float32
            dims = [int(x) for x in self.headers["x-dims"].split(",")]
            arr = np.frombuffer(body, dtype=dt).reshape(dims)
            with lock:
                out = sess.run(["flows_cellprob"], {"image": arr})[0]
            payload = np.ascontiguousarray(out, dtype="<f4").tobytes()
            self.send_response(200); self.send_header("Content-Length", str(len(payload))); self.end_headers()
            self.wfile.write(payload)
    srv = _ThreadingHTTPServer(("127.0.0.1", 0), Handler)
    threading.Thread(target=srv.serve_forever, daemon=True).start()
    return srv, srv.server_address[1]

def run_ts_cellpose_with_model(model_kind: str, model_path: str, img_rgb: np.ndarray, bridge_url: str | None):
    H, W = img_rgb.shape[:2]
    rgba = np.ascontiguousarray(np.dstack([img_rgb, np.full((H, W), 255, dtype=np.uint8)]), dtype=np.uint8)
    with tempfile.TemporaryDirectory() as td:
        rgba_p, meta_p = os.path.join(td, "img.rgba"), os.path.join(td, "meta.json")
        dp_p, cp_p = os.path.join(td, "dp.f32"), os.path.join(td, "cp.f32")
        rgba.tofile(rgba_p)
        json.dump({
            "modelKind": model_kind, "modelPath": model_path, "rgbaPath": rgba_p,
            "width": int(W), "height": int(H), "outDpPath": dp_p, "outCpPath": cp_p,
            "cudaUrl": bridge_url, "numThreads": WASM_NUM_THREADS,
        }, open(meta_p, "w"))
        subprocess.run(["node", TS_CELLPOSE_RUNNER, "--input", meta_p], cwd=REPO_DIR, check=True)
        return np.fromfile(dp_p, dtype="<f4").reshape((2, H, W)), np.fromfile(cp_p, dtype="<f4").reshape((H, W))

def run_ts_postprocess(dP: np.ndarray, cp: np.ndarray, guard: float = 0.40):
    H, W = cp.shape
    with tempfile.TemporaryDirectory() as td:
        dp_p, cp_p, meta_p = os.path.join(td, "dp.f32"), os.path.join(td, "cp.f32"), os.path.join(td, "meta.json")
        out_json, out_mask = os.path.join(td, "out.json"), os.path.join(td, "mask.i32")
        np.ascontiguousarray(dP, dtype="<f4").tofile(dp_p)
        np.ascontiguousarray(cp, dtype="<f4").tofile(cp_p)
        json.dump({
            "dPPath": dp_p, "cellprobPath": cp_p, "width": int(W), "height": int(H),
            "maxGrainAreaFraction": float(guard), "outputJsonPath": out_json, "outputMaskPath": out_mask,
        }, open(meta_p, "w"))
        subprocess.run(["node", TS_POSTPROCESS_RUNNER, "--input", meta_p], cwd=REPO_DIR, check=True)
        return np.fromfile(out_mask, dtype="<i4").reshape((H, W)), json.load(open(out_json))["grains"]

all_records = []
fp16_masks_cache = {} # (model, uid) -> ndarray

# Ensure fp16 runs first so fp16_masks_cache is populated for mask_pq_vs_fp16
ordered_variants = ["fp16"] + [v for v in VARIANTS_TO_RUN if v not in ("fp32", "fp16")]
for m in MODELS_TO_RUN:
    mask_cache_dir = os.path.join(OUT_DIR, "masks", m)
    os.makedirs(mask_cache_dir, exist_ok=True)
    for v in ordered_variants:
        if (m, v) not in variant_meta:
            continue
        t_rec = tile_results[(m, v)]
        if not t_rec["wasm"].get("run_ok"):
            print(f"Skipping 40-image eval for {m}/{v} (failed ORT-Web WASM load/run).")
            continue
        expected_dt = "float16" if m == "cpsam" else "float32"
        if variant_meta[(m, v)]["input_dtype"] != expected_dt:
            print(f"Skipping 40-image eval for {m}/{v} (input_dtype={variant_meta[(m, v)]['input_dtype']} != {expected_dt}).")
            continue

        var_csv = os.path.join(OUT_DIR, f"eval_{m}_{v}.csv")
        drive_var_csv = os.path.join(DRIVE_OUT_DIR, "cache", f"eval_{m}_{v}.csv") if DRIVE_MOUNTED else None
        if DRIVE_MOUNTED and SKIP_CACHED_ON_DRIVE and os.path.exists(drive_var_csv) and not os.path.exists(var_csv):
            shutil.copy2(drive_var_csv, var_csv)

        existing = {}
        if os.path.exists(var_csv):
            for _, row in pd.read_csv(var_csv).iterrows():
                existing[row["uid"]] = row.to_dict()

        srv, bridge_url = None, None
        if t_rec.get("bridge_matches_wasm"):
            sess = ort.InferenceSession(variant_meta[(m, v)]["path"], providers=["CPUExecutionProvider"])
            srv, port = start_ort_bridge(sess)
            bridge_url = f"http://127.0.0.1:{port}/{m}"

        var_rows = []
        try:
            for r in tqdm(test_rows, desc=f"{m}/{v} ({'CPU-bridge' if bridge_url else 'WASM'})"):
                uid = r["uid"]
                m_path = os.path.join(mask_cache_dir, f"{v}_{uid}.npy")
                if uid in existing and (v != "fp16" or os.path.exists(m_path)):
                    var_rows.append(existing[uid])
                    if v == "fp16":
                        fp16_masks_cache[(m, uid)] = np.load(m_path)
                    continue
                rgb = load_rgb(r["image_path"])
                H, W = rgb.shape[:2]
                gt_masks = load_gt_mask(r["mask_path"], (H, W))
                gt_grains = compute_mask_grain_metrics(gt_masks)
                gt_b = area_weighted_percentiles(gt_grains, "bAxisPx")

                dP, cp = run_ts_cellpose_with_model(m, variant_meta[(m, v)]["path"], rgb, bridge_url)
                ts_masks, ts_grains = run_ts_postprocess(dP, cp, 0.40)
                if v == "fp16":
                    np.save(m_path, ts_masks)
                    fp16_masks_cache[(m, uid)] = ts_masks
                    pq_fp16 = 1.0
                else:
                    pq_fp16 = compute_pq(ts_masks, fp16_masks_cache[(m, uid)]) if (m, uid) in fp16_masks_cache else float("nan")

                ts_int = [g for g in ts_grains if not g.get("touchesBorder", g.get("isBorder", False))]
                ts_b = area_weighted_percentiles(ts_int, "bAxisPx")
                var_rows.append({
                    "uid": uid, "site": r["site"], "model": m, "variant": v,
                    "n_gt": len(gt_grains), "n_pred": len(ts_int),
                    "grain_count_ratio": len(ts_int) / max(1, len(gt_grains)),
                    "gt_b_D16": gt_b["D16"], "gt_b_D50": gt_b["D50"], "gt_b_D84": gt_b["D84"],
                    "pred_b_D16": ts_b["D16"], "pred_b_D50": ts_b["D50"], "pred_b_D84": ts_b["D84"],
                    "pq_vs_gt": compute_pq(ts_masks, gt_masks),
                    "mask_pq_vs_fp16": pq_fp16,
                })
                pd.DataFrame(var_rows).to_csv(var_csv, index=False)
                if DRIVE_MOUNTED:
                    shutil.copy2(var_csv, drive_var_csv)
        finally:
            if srv is not None:
                srv.shutdown()
        all_records.extend(var_rows)

df_all = pd.DataFrame(all_records)
per_img_out = os.path.join(OUT_DIR, "results_mixed_w4_w8_per_image.csv")
df_all.to_csv(per_img_out, index=False)
if DRIVE_MOUNTED:
    shutil.copy2(per_img_out, os.path.join(DRIVE_OUT_DIR, "results_mixed_w4_w8_per_image.csv"))
print("Saved results_mixed_w4_w8_per_image.csv:", len(df_all), "rows")


In [ ]:
# Cell 6: Evaluate pass criteria, generate results_mixed_w4_w8.json, and print summary table
import json, os, shutil
import numpy as np
import pandas as pd

def bootstrap_ci(arr: np.ndarray, fn, n_boot: int = 2000, seed: int = SEED) -> list[float]:
    arr = arr[np.isfinite(arr)]
    if len(arr) == 0:
        return [float("nan"), float("nan")]
    if len(arr) == 1:
        v = float(fn(arr)); return [v, v]
    rng = np.random.default_rng(seed)
    boots = np.array([fn(arr[rng.integers(0, len(arr), size=len(arr))]) for _ in range(n_boot)], dtype=np.float64)
    return [float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))]

def summarize_ape(sub: pd.DataFrame, gt_col: str, pred_col: str) -> dict:
    valid = sub[(sub["n_gt"] > 0) & sub[gt_col].notna() & sub[pred_col].notna() & (sub[gt_col] > 0)]
    if len(valid) == 0:
        return {"n": 0}
    ape = 100.0 * np.abs(valid[pred_col].to_numpy() - valid[gt_col].to_numpy()) / valid[gt_col].to_numpy()
    return {
        "n": int(len(valid)),
        "mean_ape_pct": float(np.mean(ape)),
        "mean_ape_ci95": bootstrap_ci(ape, np.mean),
        "median_ape_pct": float(np.median(ape)),
        "median_ape_ci95": bootstrap_ci(ape, np.median),
    }

results = {"seed": SEED, "eval_images": len(test_rows), "models": {}}

for m in MODELS_TO_RUN:
    results["models"][m] = {}
    fp16_sub = df_all[(df_all["model"] == m) & (df_all["variant"] == "fp16")] if len(df_all) else pd.DataFrame()
    fp16_pq = float(fp16_sub["pq_vs_gt"].mean()) if len(fp16_sub) else None
    fp16_d50_med = summarize_ape(fp16_sub, "gt_b_D50", "pred_b_D50").get("median_ape_pct") if len(fp16_sub) else None

    for v in VARIANTS_TO_RUN:
        if (m, v) not in variant_meta:
            continue
        vm = variant_meta[(m, v)]
        tr = tile_results[(m, v)]
        sub = df_all[(df_all["model"] == m) & (df_all["variant"] == v)] if len(df_all) else pd.DataFrame()

        eval_summary = None
        if len(sub):
            pq_arr = sub["pq_vs_gt"].to_numpy(dtype=np.float64)
            gcr_arr = sub["grain_count_ratio"].to_numpy(dtype=np.float64)
            mpq_arr = sub["mask_pq_vs_fp16"].to_numpy(dtype=np.float64)
            eval_summary = {
                "pq_vs_gt_mean": float(np.mean(pq_arr)),
                "pq_vs_gt_ci95": bootstrap_ci(pq_arr, np.mean),
                "D16": summarize_ape(sub, "gt_b_D16", "pred_b_D16"),
                "D50": summarize_ape(sub, "gt_b_D50", "pred_b_D50"),
                "D84": summarize_ape(sub, "gt_b_D84", "pred_b_D84"),
                "grain_count_ratio_mean": float(np.mean(gcr_arr)),
                "grain_count_ratio_ci95": bootstrap_ci(gcr_arr, np.mean),
                "mask_pq_vs_fp16_mean": float(np.nanmean(mpq_arr)),
            }

        # Pass criteria
        loads_wasm = bool(tr["wasm"].get("load_ok") and tr["wasm"].get("run_ok"))
        num = tr.get("numerics_vs_pt_fp32") or {}
        numerics_ok = bool(num.get("non_finite_count", 1) == 0 and num.get("all_zero_negative_ok", False))
        acc_ok = False
        mask_ok = False
        if eval_summary and fp16_pq is not None and fp16_d50_med is not None:
            acc_ok = (abs(eval_summary["pq_vs_gt_mean"] - fp16_pq) <= 0.01) and (abs(eval_summary["D50"]["median_ape_pct"] - fp16_d50_med) <= 1.0)
            mask_ok = eval_summary["mask_pq_vs_fp16_mean"] >= 0.95
        passed = loads_wasm and numerics_ok and (True if v == "fp32" else (acc_ok and mask_ok))

        results["models"][m][v] = {
            "size": vm["size"],
            "quantized_layer_count": vm["quantized_layer_count"],
            "quantized_weight_fraction": vm["quantized_weight_fraction"],
            "wasm": tr["wasm"],
            "speed_ms": tr["speed_ms"],
            "numerics_vs_pt_fp32": num,
            "heldout_40": eval_summary,
            "pass_criteria": {
                "loads_and_runs_wasm": loads_wasm,
                "finite_and_allzero_negative": numerics_ok,
                "pq_and_d50_within_tol": acc_ok if v != "fp32" else None,
                "mask_pq_vs_fp16_ge_095": mask_ok if v != "fp32" else None,
                "PASS": passed,
            },
        }

res_path = os.path.join(OUT_DIR, "results_mixed_w4_w8.json")
with open(res_path, "w") as f:
    json.dump(results, f, indent=2)
if DRIVE_MOUNTED:
    shutil.copy2(res_path, os.path.join(DRIVE_OUT_DIR, "results_mixed_w4_w8.json"))

print("\n=== QUANTIZATION SUMMARY ===")
for m, vars_d in results["models"].items():
    print(f"\nModel: {m}")
    for v, d in vars_d.items():
        h = d["heldout_40"] or {}
        d50 = (h.get("D50") or {}).get("median_ape_pct")
        pq = h.get("pq_vs_gt_mean")
        mpq = h.get("mask_pq_vs_fp16_mean")
        print(f"  {v:14s} | {d['size']['mb']:6.1f} MB | WASM={'OK' if d['pass_criteria']['loads_and_runs_wasm'] else 'FAIL'} | PQ={pq if pq is not None else 'n/a'} | D50_med={d50 if d50 is not None else 'n/a'} | maskPQ_fp16={mpq if mpq is not None else 'n/a'} | PASS={d['pass_criteria']['PASS']}")
